# 第 28 天：ML选股

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：ML选股
> 必做：预测收益
> 选做：组合构建
> 目标产出：ML选股策略

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 把模型预测值变成股票打分。
2. 构建多空组合、只多组合和行业中性组合。
3. 检查组合收益、换手、暴露和回撤。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

预测不是终点，组合才是落地。一个预测 IC 不错的模型，如果换手太高、行业暴露失控、回撤难看，仍然不是好策略。

## 5. 今日核心实验


### 实验 1：训练一个简单 ML 打分模型

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
feature_names = ["value", "quality", "growth", "momentum_20", "momentum_60", "low_vol", "liquidity", "reversal_5", "price_volume"]
panel = build_panel({name: factor_library[name] for name in feature_names}, future_5d)
panel["value_x_quality"] = panel["value"] * panel["quality"]
panel["short_long_mom_gap"] = panel["momentum_20"] - panel["momentum_60"]
train, test, split_date = time_split_panel(panel, 0.7)
features = [c for c in panel.columns if c != "label"]

coef = fit_ridge(train[features].to_numpy(), train["label"].to_numpy(), lam=10.0)
test_score = pd.Series(
    predict_ridge(test[features].to_numpy(), coef),
    index=test.index,
    name="ml_score",
).unstack("asset")

print("ML score 矩阵：", test_score.shape)


### 实验 2：多空组合：先看排序能力能否变成收益

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
test_next_return = returns.reindex(test_score.index)
ml_weights = make_market_neutral_weights(test_score, q=0.2)
ml_strategy_ret = portfolio_return(ml_weights, test_next_return)

print(ml_strategy_ret.describe().round(5))


### 实验 3：只多组合：更贴近很多真实资金约束

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
rank_pct = test_score.rank(axis=1, pct=True)
long_only = rank_pct.ge(0.8).astype(float)
long_only_weights = long_only.div(long_only.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
long_only_ret = portfolio_return(long_only_weights, test_next_return)

compare_ret = pd.DataFrame({
    "long_short": ml_strategy_ret,
    "long_only": long_only_ret,
}).dropna()

print(compare_ret.describe().round(5))


### 实验 4：回撤和换手：策略不是只看收益

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
def max_drawdown(ret: pd.Series) -> float:
    nav = (1 + ret.fillna(0)).cumprod()
    drawdown = nav / nav.cummax() - 1
    return drawdown.min()

stats = pd.DataFrame({
    "mean": compare_ret.mean(),
    "vol": compare_ret.std(),
    "sharpe_like": compare_ret.mean() / compare_ret.std() * np.sqrt(252),
    "max_drawdown": compare_ret.apply(max_drawdown),
    "turnover": pd.Series({
        "long_short": ml_weights.diff().abs().sum(axis=1).mean(),
        "long_only": long_only_weights.diff().abs().sum(axis=1).mean(),
    }),
})

print(stats.round(4))


### 实验 5：策略净值图：一眼看样本外体验

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
(1 + compare_ret.fillna(0)).cumprod().plot(figsize=(10, 4), title="ML 选股策略样本外净值")
plt.axhline(1, color="black", linewidth=1)
plt.tight_layout()
plt.show()
plt.close()


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：ML选股
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：预测值未经标准化直接拿来配权。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：只看收益曲线，不看持仓集中度和换手。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：模型预测强但组合约束差，最后交易结果变形。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：忽略样本外滚动训练，只做一次固定训练。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 29 天会整理 Alpha Zoo，把所有因子纳入可维护仓库。

## 13. 一句话收尾

ML选股 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒（更新版）

今天如果只记住一件事，记住这个：**预测IC高≠策略能赚钱。** 换手率、交易成本和行业暴露会吃掉所有Alpha。

---

## 15. 进阶：交易成本模型 —— 从理论收益到实际收益

> 课程中的回测假设零成本交易。现实中，印花税、佣金、滑点和冲击成本会显著侵蚀收益。

### 15.1 A股交易成本速查

| 费用项 | 比率 | 说明 |
|--------|------|------|
| 印花税 | 0.05%（卖出单边） | 2023年8月起减半 |
| 佣金 | 0.01%-0.03% | 双边收取 |
| 过户费 | 0.001% | 双边 |
| 滑点 | 0.05%-0.2% | 取决于流动性和下单量 |
| 冲击成本 | 0.1%-0.5% | 大单对市场价格的推动 |

**保守估计：单次换手成本约 0.15%-0.3%（双边）。**

### 15.2 成本调整组合收益


In [ ]:
def cost_adjusted_return(
    weights: pd.DataFrame,
    next_return: pd.DataFrame,
    cost_per_trade: float = 0.002,  # 单边0.2%
) -> pd.Series:
    """
    计算扣除交易成本后的组合收益。
    
    参数：
    - weights: (T × N) 每期目标权重
    - next_return: (T × N) 下一期收益率
    - cost_per_trade: 单边交易成本率
    """
    # 换手 = 权重变化的绝对值之和 / 2（因为买卖各占一半）
    turnover = weights.diff().abs().sum(axis=1) / 2
    
    # 原始收益
    raw_ret = (weights.shift(1) * next_return).sum(axis=1)
    
    # 扣除交易成本
    cost = turnover.shift(1) * cost_per_trade
    net_ret = raw_ret - cost
    
    return net_ret.dropna()

# 使用
# net_returns = cost_adjusted_return(portfolio_weights, daily_returns, cost_per_trade=0.002)
# print(f"年化收益（扣成本前）: {raw_ret.mean() * 252:.2%}")
# print(f"年化收益（扣成本后）: {net_ret.mean() * 252:.2%}")
# print(f"成本侵蚀: {(raw_ret.mean() - net_ret.mean()) * 252:.2%}")


### 15.3 换手率约束


In [ ]:
def turnover_constraint(weights: pd.DataFrame, max_turnover: float = 0.5):
    """
    检查换手率是否超过约束。
    max_turnover=0.5 表示单边换手不超过50%
    """
    turnover = weights.diff().abs().sum(axis=1) / 2
    violation_days = (turnover > max_turnover).sum()
    total_days = len(turnover.dropna())
    
    print(f"换手率统计：")
    print(f"  均值: {turnover.mean():.2%}")
    print(f"  中位数: {turnover.median():.2%}")
    print(f"  最大: {turnover.max():.2%}")
    print(f"  超标天数: {violation_days}/{total_days} ({violation_days/total_days:.1%})")
    
    return turnover

# turnover_series = turnover_constraint(portfolio_weights, max_turnover=0.5)


---

## 16. 进阶：组合优化基础 —— 从打分到权重

> 简单的等权或分位数加权容易导致行业集中和风格暴露。组合优化可以显式控制风险。

### 16.1 均值-方差优化（简化版）


In [ ]:
from scipy.optimize import minimize

def mean_variance_weights(
    expected_returns: np.ndarray,
    cov_matrix: np.ndarray,
    risk_aversion: float = 1.0,
    long_only: bool = True,
):
    """
    简单均值-方差优化。
    
    参数：
    - expected_returns: (N,) 预期收益向量
    - cov_matrix: (N, N) 协方差矩阵
    - risk_aversion: 风险厌恶系数（越大越保守）
    - long_only: 是否只做多
    """
    n = len(expected_returns)
    
    def objective(w):
        # 最大化: w^T * mu - lambda * w^T * Sigma * w
        return -(w @ expected_returns - risk_aversion * w @ cov_matrix @ w)
    
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0, 1) if long_only else (-1, 1) for _ in range(n)]
    
    result = minimize(
        objective,
        x0=np.ones(n) / n,  # 等权初始
        bounds=bounds,
        constraints=constraints,
        method="SLSQP",
    )
    
    return result.x

# 示例
# expected_ret = np.array([0.05, 0.03, 0.02, 0.04, 0.01])
# cov = np.cov(returns_matrix, rowvar=False)  # 历史协方差
# w = mean_variance_weights(expected_ret, cov, risk_aversion=2.0)
# print(f"优化权重: {w}")


### 16.2 行业中性约束


In [ ]:
def industry_neutral_weights(
    scores: pd.Series,
    industry_map: pd.Series,
    top_pct: float = 0.2,
):
    """
    在每个行业内独立选股，构建行业中性组合。
    """
    weights = pd.Series(0.0, index=scores.index)
    n_industries = industry_map.nunique()
    
    for ind, members in industry_map.groupby(industry_map).groups.items():
        ind_scores = scores.loc[list(members)].dropna()
        n_select = max(1, int(len(ind_scores) * top_pct))
        selected = ind_scores.nlargest(n_select).index
        weights.loc[selected] = 1.0 / (n_industries * n_select)
    
    return weights / weights.sum()  # 确保权重和为1

# ind_weights = industry_neutral_weights(ml_score.iloc[-1], industries)
# print(f"行业暴露: {ind_weights.groupby(industries).sum()}")


---

## 17. 进阶：回撤与风险分析

### 17.1 核心风险指标


In [ ]:
def risk_metrics(returns: pd.Series, rf: float = 0.02) -> dict:
    """
    计算核心风险指标。
    rf: 年化无风险利率（默认2%）
    """
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - rf) / ann_vol if ann_vol > 0 else 0
    
    # 最大回撤
    cum = (1 + returns).cumprod()
    peak = cum.expanding().max()
    drawdown = (cum - peak) / peak
    max_dd = drawdown.min()
    
    # Calmar比率
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    
    # 胜率
    win_rate = (returns > 0).mean()
    
    # 盈亏比
    avg_win = returns[returns > 0].mean()
    avg_loss = abs(returns[returns < 0].mean())
    profit_factor = avg_win / avg_loss if avg_loss > 0 else float("inf")
    
    return {
        "年化收益": f"{ann_ret:.2%}",
        "年化波动": f"{ann_vol:.2%}",
        "Sharpe比率": round(sharpe, 2),
        "最大回撤": f"{max_dd:.2%}",
        "Calmar比率": round(calmar, 2),
        "胜率": f"{win_rate:.1%}",
        "盈亏比": round(profit_factor, 2),
    }

# metrics = risk_metrics(net_returns)
# for k, v in metrics.items():
#     print(f"{k}: {v}")


### 17.2 滚动回撤可视化


In [ ]:
def plot_rolling_drawdown(returns: pd.Series, window: int = 252):
    """绘制滚动回撤"""
    cum = (1 + returns).cumprod()
    rolling_peak = cum.rolling(window, min_periods=1).max()
    rolling_dd = (cum - rolling_peak) / rolling_peak
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # 净值曲线
    axes[0].plot(cum.index, cum.values)
    axes[0].set_title("累计净值")
    axes[0].grid(True)
    
    # 回撤
    axes[1].fill_between(
        rolling_dd.index, 0, rolling_dd.values,
        alpha=0.3, color="red",
    )
    axes[1].plot(rolling_dd.index, rolling_dd.values, color="red")
    axes[1].set_title(f"滚动{window}日最大回撤")
    axes[1].grid(True)
    
    plt.tight_layout()
    return fig

# fig = plot_rolling_drawdown(net_returns)


---

## 18. 进阶：策略容量估算


In [ ]:
def estimate_capacity(
    daily_volume: pd.DataFrame,
    weights: pd.DataFrame,
    participation_rate: float = 0.05,
):
    """
    粗略估算策略容量。
    
    参数：
    - daily_volume: (T × N) 日均成交额
    - weights: (T × N) 目标权重
    - participation_rate: 目标参与率（不超过日成交额的百分比）
    
    返回：每日最大可投金额（元）
    """
    # 每只股票的容量 = 日成交额 × 参与率
    stock_capacity = daily_volume * participation_rate
    
    # 权重不为0的股票的最小容量 / 权重 = 组合容量
    aligned_volume = stock_capacity.reindex_like(weights)
    aligned_weights = weights.abs()
    
    # 对每期：capacity = min(stock_capacity_i / weight_i) 对所有持仓股票
    capacity_per_day = (aligned_volume / aligned_weights.replace(0, np.nan)).min(axis=1)
    
    # 取中位数作为稳健估计
    median_capacity = capacity_per_day.median()
    
    print(f"策略容量估计（参与率={participation_rate:.0%}）：")
    print(f"  中位数: ¥{median_capacity:,.0f}")
    print(f"  25分位: ¥{capacity_per_day.quantile(0.25):,.0f}")
    print(f"  75分位: ¥{capacity_per_day.quantile(0.75):,.0f}")
    
    return capacity_per_day

# capacity = estimate_capacity(volume_panel, portfolio_weights, participation_rate=0.05)


---

本课程内容仅用于量化研究学习，不构成投资建议。
